# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Convert metadata to JSON-compatible dict
metadata = dataset.metadata.to_json()

print(f"{metadata['name']}: {metadata['description']}")
print(f"Version: {metadata['version']}")
print(f"Published: {metadata['datePublished']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's list all `@id`s for record sets, fields, and columns, as defined in the dataset's Croissant metadata.

In [ ]:
# The Croissant Dataset exposes accessible record set IDs and field/column details through `metadata`.

def get_record_sets(meta):
    # record sets are under 'recordSet' as an array
    if 'recordSet' in meta and isinstance(meta['recordSet'], list):
        return meta['recordSet']
    elif 'recordSet' in meta and isinstance(meta['recordSet'], dict):
        return [meta['recordSet']]
    else:
        return []

record_sets = get_record_sets(metadata)
if not record_sets:
    print("No record sets found directly in metadata['recordSet']. Attempting to infer from distributions...")
    # fallback option: If direct recordSets are not present, infer from file objects in 'distribution'
    # mlcroissant allows discovering record set IDs via API
    record_set_ids = list(dataset.record_sets.keys())
    print(f"Discovered record set @id's: {record_set_ids}")
else:
    record_set_ids = [rs['@id'] for rs in record_sets]
    print(f"Discovered record set @id's: {record_set_ids}")

# For each record set, print info and field/column structure
print("\nRecord set structural overview:")
for rs_id in record_set_ids:
    rec_set = dataset.record_sets[rs_id]
    print(f"RecordSet @id: {rs_id}")
    print(f"  Name: {rec_set.metadata.get('name', 'No name specified')}")
    # Print field and columns for the record set
    # fields are at rec_set.metadata['field'] (list or dict)
    fields = rec_set.metadata.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        print(f"    Field @id: {field['@id']} (name: {field.get('name', '')}, dataType: {field.get('dataType', '')})")
        # columns are under 'column', could be ID, dict, or list
        cols = field.get('column', [])
        if isinstance(cols, dict):
            cols = [cols]
        elif isinstance(cols, str): # single column by @id
            cols = [cols]
        for col in cols:
            if isinstance(col, dict):
                print(f"      Column @id: {col['@id']} (name: {col.get('name', '')})")
            else:
                print(f"      Column @id: {col}")
    print()


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# To demonstrate, we'll extract all available record sets and load them into pandas DataFrames.

# If record_set_ids is empty, an issue occurred above; otherwise, proceed.
dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nLoading records for RecordSet @id: {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"  No records available for {record_set_id}")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Loaded {len(df)} records for {record_set_id}")
    print(f"  Columns: {df.columns.tolist()}")
    display(df.head())

# For clarity, select a main record set for further EDA. If only one, we use that. Otherwise, pick the most populated or the first.
if dataframes:
    primary_record_set_id = sorted(dataframes, key=lambda k: -len(dataframes[k]))[0]
    print(f"\nPrimary record set for analysis: {primary_record_set_id}")
else:
    print("No data could be extracted from any record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

if dataframes:
    df = dataframes[primary_record_set_id]
    print(f"Working on primary record set: {primary_record_set_id}")
    # Try to automatically detect numeric fields for demonstration
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_columns:
        # Try to convert any columns that look like numbers
        for c in df.columns:
            try:
                df[c] = pd.to_numeric(df[c])
            except Exception:
                continue
        numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Detected numeric fields: {numeric_columns}")
    if not numeric_columns:
        print("No numeric field detected; skipping filtering and normalization example.")
    else:
        # Use the first detected numeric field
        numeric_field = numeric_columns[0]
        print(f"Using {numeric_field} as the numeric field for filtering and normalization.")

        threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile as demonstration
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a likely categorical/demographic column (e.g., 'gender' or similar)
        group_fields = [c for c in df.columns if ('gender' in c.lower() or 'education' in c.lower() or 'age' in c.lower()) and not c==numeric_field]
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
            print(f"Grouped mean of {numeric_field} by '{group_field}':")
            display(grouped_df)
        else:
            print("No common group field (e.g., gender, age, education) detected for grouping.")
else:
    print("No DataFrame to perform EDA on.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_columns:
    # Histogram for the selected numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot grouped by categorical field if possible
    if group_fields:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


In this notebook, we demonstrated how to load, explore, and perform initial analysis of the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. The dataset provides a rich resource for studying demographic and mental health indicators, and the Croissant schema enables clear, reproducible data extraction.

**Key steps performed:**
- Loaded dataset structure and metadata via Croissant schema
- Inspected available record sets, fields, and column structure via their `@id`s
- Loaded records into pandas DataFrames for analysis
- Performed basic EDA: filtering, normalization, and grouping by demographic variables
- Visualized numeric distributions and group differences

This workflow can be adapted to any dataset exposing a Croissant JSON-LD schema, and extended for further statistical analysis or machine learning.

_For more, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/)_